[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C02_Post_Training_Course/05_rlvr_grpo/05_rlvr_grpo.ipynb)

# 05 · 推理模型与 RLVR/GRPO —— 手写 GRPO 完整循环

> **MODULE 05 / 8** · 配套讲解：[05_讲解.html](05_讲解.html) · 纯 PyTorch + matplotlib，**CPU 全程 < 1 分钟**

本 notebook 在一个**玩具可验证任务**（两位数加法）上，从零实现 GRPO 的全部要素，并复现讲解里的关键现象：

1. **任务与 verifier**：参数化答案分布（小 MLP）+ 精确匹配 verifier；
2. **GRPO 完整循环**：每题采 G=8 个答案 → verifier 给 0/1 reward → 组内标准化 advantage → policy gradient + KL 正则，准确率从随机（~0.5%）升到 **>90%**；
3. **退化演示**：组内全对/全错时 std=0，naive 实现产生 NaN；ε 与 skip 两种处理对比；顺带观察 **entropy collapse**；
4. **format reward**：reward = accuracy·format + 0.2·format，复现"先学格式、再学正确"的两阶段曲线；
5. **拒绝采样微调（RFT）对比**：BoN 采样 + SFT 路线，同任务比样本效率；
6. **overthinking 玩具**：给奖励加"长度成本"后，策略主动缩短输出；
7. ✏️ 三道练习 + 📖 参考答案。

## 玩具任务设计：与真实 RLVR 的对应

| 真实 RLVR（如 DeepSeek-R1） | 本 notebook |
|---|---|
| LLM 自回归生成解答 | 小 MLP 把题目特征映射成**答案分布**（一次采样出整个"答案"） |
| 题目：数学/代码题 | 题目：$a+b$，$a,b\in[10,99]$ |
| 答案空间：自由文本 | 答案空间：$\{0,1,\dots,198\}$（199 类，覆盖所有可能的和 20..198） |
| verifier：答案提取 + 精确比对 / 跑测试 | verifier：`answer == a + b`（精确匹配，0/1） |
| 组采样 G 条 rollout | 每题从答案分布采 G=8 个答案 |
| KL 锚定 SFT 后的参考模型 | KL 锚定随机初始化的冻结参考网络 |

把"生成一条序列"压缩成"采样一个类别"，去掉了逐 token 的复杂度，但**保留了 GRPO 的全部统计结构**：稀疏 0/1 奖励、组内相对 advantage、KL 正则、探索-利用权衡。

In [ ]:
import copy
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
N_ANS = 199   # 答案词表：0..198（两位数之和最大 99+99=198）

def make_questions(n, gen=None):
    # 随机生成 n 道两位数加法题
    a = torch.randint(10, 100, (n,), generator=gen)
    b = torch.randint(10, 100, (n,), generator=gen)
    return a, b

def features(a, b):
    # 题目编码：4 个数位各自 one-hot（十位 10 维 + 个位 10 维）x2 = 40 维
    # （one-hot 数位让 MLP 容易学到进位结构；连续编码会卡在 ~85%，可自行实验）
    return torch.cat([F.one_hot(a // 10, 10), F.one_hot(a % 10, 10),
                      F.one_hot(b // 10, 10), F.one_hot(b % 10, 10)], dim=-1).float()

class PolicyNet(nn.Module):
    # "策略" = 题目特征 -> 答案 logits（n_out=N_ANS；format 实验会加 2 维 format 头）
    def __init__(self, hidden=128, n_out=N_ANS):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(40, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, n_out))
    def forward(self, x):
        return self.net(x)

def verify_batch(a, b, answers):
    # 张量化 verifier：精确匹配 -> 0/1 奖励。answers: (B, G)
    return (answers == (a + b).unsqueeze(1)).float()

def greedy_accuracy(policy, n=2000):
    # 固定种子的 held-out 题集上，贪心解码（argmax）准确率
    g = torch.Generator().manual_seed(123)
    a, b = make_questions(n, g)
    with torch.no_grad():
        pred = policy(features(a, b))[:, :N_ANS].argmax(-1)
    return (pred == a + b).float().mean().item()

rand_policy = PolicyNet()
print(f"随机初始化策略的贪心准确率: {greedy_accuracy(rand_policy):.4f}  (随机水平 ~ 1/199 = {1/199:.4f})")
a, b = make_questions(4)
ans = torch.distributions.Categorical(logits=rand_policy(features(a, b))).sample((8,)).T
print("示例题目:", [(int(x), int(y), int(x + y)) for x, y in zip(a, b)][:2])
print("每题采 8 个答案 (前 2 题):", ans[:2].tolist())
print("verifier 奖励 (前 2 题):", verify_batch(a, b, ans)[:2].tolist())

## GRPO 核心循环

对每道题 $q$ 采一组 $G$ 个答案，verifier 给奖励 $r_i \in \{0,1\}$，**组内相对 advantage**：

$$\hat{A}_i = \frac{r_i - \mathrm{mean}(r_1,\dots,r_G)}{\mathrm{std}(r_1,\dots,r_G) + \varepsilon}$$

损失（无 critic！）：

$$\mathcal{L} = -\frac{1}{BG}\sum_{i} \hat{A}_i \log \pi_\theta(o_i \mid q_i) \;+\; \beta\,\widehat{\mathbb{D}}_{KL}\!\left[\pi_\theta \,\|\, \pi_{\text{ref}}\right] \;-\; c_{\text{ent}}\,\mathcal{H}[\pi_\theta]$$

三点说明（对应讲解 §2）：

- **importance ratio 与 clip 哪去了？** 我们每批采样后只做一次梯度更新（严格 on-policy），此时 $\rho_{i,t} = \pi_\theta/\pi_{\theta_{old}} = 1$，min/clip 全部退化，目标化简为 REINFORCE 式 $-\hat A_i \log\pi_\theta$。真实 GRPO 在同一批数据上做 $\mu$ 次内层更新，第 2 次起 $\rho \ne 1$，clip 才生效。
- **KL 用 $k_3$ 无偏估计器**：$\widehat{\mathbb{D}}_{KL} = \frac{\pi_{\text{ref}}}{\pi_\theta} - \log\frac{\pi_{\text{ref}}}{\pi_\theta} - 1 \ge 0$，只在采到的样本上计算，无需遍历词表。
- **entropy bonus（$c_{\text{ent}}$）是本玩具的"探索假肢"**：真实 RLVR 从预训练模型起步，base model 的分布本身就是强探索先验（"RL 放大已有的低概率正确行为"）；我们从随机初始化起步，没有这个先验，不加 entropy bonus 策略会过早变得"自信地错"——组内全错 → std=0 → 零梯度 → 永远卡住。下一节会把这条失败曲线跑给你看。

In [ ]:
def group_advantage(rewards, eps=1e-4):
    # rewards: (B, G) -> 组内标准化 advantage（总体标准差 unbiased=False）
    mean = rewards.mean(dim=-1, keepdim=True)
    std = rewards.std(dim=-1, unbiased=False, keepdim=True)
    return (rewards - mean) / (std + eps)

def grpo_loss(logp, advantages, logp_ref, beta=0.0):
    # policy gradient 项 + beta * KL(pi_theta || pi_ref) 的 k3 估计
    pg = -(advantages * logp).mean()
    log_ratio = logp_ref - logp                    # log(pi_ref / pi_theta)，逐样本
    kl = (log_ratio.exp() - log_ratio - 1).mean()  # k3：无偏、低方差、恒 >= 0
    return pg + beta * kl

def train_grpo(steps=5000, B=128, G=8, beta=0.005, lr=3e-3, ent_coef=0.10,
               seed=0, eval_every=250):
    torch.manual_seed(seed)
    policy = PolicyNet()
    ref = copy.deepcopy(policy)                    # 冻结参考策略（KL 锚点）
    for p in ref.parameters():
        p.requires_grad_(False)
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    hist = []
    for step in range(1, steps + 1):
        a, b = make_questions(B)
        x = features(a, b)
        logits = policy(x)
        logp_all = F.log_softmax(logits, -1)
        with torch.no_grad():                      # 采样不带梯度
            samp = torch.distributions.Categorical(logits=logits).sample((G,)).T  # (B,G)
        r = verify_batch(a, b, samp)               # 0/1 奖励
        adv = group_advantage(r)                   # 组内相对 advantage
        logp = logp_all.gather(1, samp)            # (B,G) 采样答案的 log prob（带梯度）
        with torch.no_grad():
            logp_ref = F.log_softmax(ref(x), -1).gather(1, samp)
        entropy = -(logp_all.exp() * logp_all).sum(-1).mean()
        loss = grpo_loss(logp, adv, logp_ref, beta) - ent_coef * entropy
        opt.zero_grad(); loss.backward(); opt.step()
        if step % eval_every == 0:
            hist.append(dict(step=step, samples=step * B * G,
                             acc=greedy_accuracy(policy),
                             batch_reward=r.mean().item(), entropy=entropy.item()))
    return policy, hist

policy_grpo, hist_grpo = train_grpo()
print(f"最终贪心准确率: {hist_grpo[-1]['acc']:.3f}  (目标 > 0.9)")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot([h['step'] for h in hist_grpo], [h['acc'] for h in hist_grpo], marker='.')
axes[0].axhline(0.9, ls='--', c='gray', lw=0.8)
axes[0].set(xlabel='step', ylabel='greedy accuracy', title='GRPO: accuracy')
axes[1].plot([h['step'] for h in hist_grpo], [h['entropy'] for h in hist_grpo], marker='.', c='tab:orange')
axes[1].set(xlabel='step', ylabel='policy entropy (nats)', title='GRPO: entropy')
plt.tight_layout(); plt.show()

## 组内全对/全错的退化：std=0

二值奖励下，只要一组 $G$ 个答案**全对或全错**，$\mathrm{std}(r)=0$：

- **naive 实现**（直接除以 std）→ `0/0 = NaN`，一个 NaN 经过 `mean()` 毒化整个 batch 的梯度，训练当场死亡；
- **ε 平滑**：分子 $r_i-\mathrm{mean}(r)$ 本来就是 0，加 ε 后整组 advantage = 0 —— 该题贡献零梯度（但仍参与 KL/entropy 项）;
- **skip 策略**：把退化组从 batch 中整个剔除。对 0/1 奖励，剔除的组 advantage 本来就是 0，所以 PG 项理论上与 ε 平滑等价——**但有一个归一化陷阱**：剔除后若对剩余样本取 `mean`（最顺手的写法），分母从 $BG$ 变成"保留样本数"，等于按退化比例**隐式放大有效学习率**（早期退化率 ~97%，放大 ~30 倍）。下面实验会展示这个不起眼的实现选择足以让训练从 95% 掉到 50%。

更要紧的是退化与 **entropy collapse** 的相互锁死：策略一旦"自信地错"，全组采出同一个错误答案 → std=0 → 零梯度 → 无法纠错 → 更自信。数据侧解法是难度课程（讲解 §7：筛掉 pass rate≈0/1 的题），算法侧解法是维持探索（entropy bonus）。下面把这些全部跑出来。

In [ ]:
# --- (1) naive 除以 std：NaN 毒化 ---
r_allsame = torch.tensor([[1., 1., 1., 1.], [1., 0., 0., 0.]])
naive = (r_allsame - r_allsame.mean(-1, keepdim=True)) / r_allsame.std(-1, unbiased=False, keepdim=True)
print("naive advantage:\n", naive)
print("含 NaN:", torch.isnan(naive).any().item(), "-> 任何后续 mean()/backward() 全部变 NaN\n")

# --- (2) eps 平滑：退化组 advantage 恒为 0 ---
print("eps 平滑 advantage:\n", group_advantage(r_allsame))
print("全对组 -> 全 0（零梯度）; 混合组 -> 正负分明\n")

# --- (3) 退化组比例：初始 vs 训练后 ---
def degenerate_fraction(policy, B=1024, G=8):
    a, b = make_questions(B)
    with torch.no_grad():
        samp = torch.distributions.Categorical(logits=policy(features(a, b))).sample((G,)).T
    return (verify_batch(a, b, samp).std(-1, unbiased=False) == 0).float().mean().item()

torch.manual_seed(42)
print(f"退化组比例  初始策略: {degenerate_fraction(PolicyNet()):.2%} (几乎全错)"
      f" | 训练后: {degenerate_fraction(policy_grpo):.2%} (几乎全对)")
print("-> 有效梯度集中在中间阶段; 数据课程的本质就是把题目难度维持在退化率低的区间\n")

# --- (4) 不加 entropy bonus: 自信地错 -> 卡死 ---
_, hist_noent = train_grpo(steps=3000, ent_coef=0.0)

# --- (5) skip 策略(剔除退化组), 两种归一化 ---
def train_grpo_skip(norm, steps=5000, B=128, G=8, beta=0.005, lr=3e-3, ent_coef=0.10, seed=0, eval_every=250):
    # norm="full": PG/KL 用 sum/(B*G) 归一 -> 与 eps 平滑数学等价
    # norm="kept": PG/KL 对保留样本取 mean -> 隐式放大有效学习率(常见实现陷阱)
    torch.manual_seed(seed)
    policy = PolicyNet(); ref = copy.deepcopy(policy)
    for p in ref.parameters(): p.requires_grad_(False)
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    hist = []
    for step in range(1, steps + 1):
        a, b = make_questions(B)
        x = features(a, b)
        logits = policy(x)
        logp_all = F.log_softmax(logits, -1)
        with torch.no_grad():
            samp = torch.distributions.Categorical(logits=logits).sample((G,)).T
        r = verify_batch(a, b, samp)
        keep = r.std(-1, unbiased=False) > 0            # 只保留非退化组
        if keep.sum() == 0:
            continue
        adv = group_advantage(r[keep])
        logp = logp_all[keep].gather(1, samp[keep])
        with torch.no_grad():
            logp_ref = F.log_softmax(ref(x[keep]), -1).gather(1, samp[keep])
        denom = B * G if norm == "full" else logp.numel()
        pg = -(adv * logp).sum() / denom
        log_ratio = logp_ref - logp
        kl = (log_ratio.exp() - log_ratio - 1).sum() / denom
        entropy = -(logp_all.exp() * logp_all).sum(-1).mean()  # entropy 仍用全 batch
        loss = pg + beta * kl - ent_coef * entropy
        opt.zero_grad(); loss.backward(); opt.step()
        if step % eval_every == 0:
            hist.append(dict(step=step, acc=greedy_accuracy(policy)))
    return hist

hist_skip_full = train_grpo_skip("full")
hist_skip_kept = train_grpo_skip("kept")
plt.figure(figsize=(6.8, 3.4))
plt.plot([h['step'] for h in hist_grpo], [h['acc'] for h in hist_grpo], label='eps smoothing')
plt.plot([h['step'] for h in hist_skip_full], [h['acc'] for h in hist_skip_full], ls='--', label='skip, sum/(B*G)')
plt.plot([h['step'] for h in hist_skip_kept], [h['acc'] for h in hist_skip_kept], label='skip, mean over kept (trap!)')
plt.plot([h['step'] for h in hist_noent], [h['acc'] for h in hist_noent], label='eps, NO entropy bonus')
plt.xlabel('step'); plt.ylabel('greedy accuracy'); plt.legend(fontsize=8); plt.title('degenerate handling & entropy collapse')
plt.tight_layout(); plt.show()
print(f"eps {hist_grpo[-1]['acc']:.3f} | skip(full) {hist_skip_full[-1]['acc']:.3f} -> 二者等价;")
print(f"skip(mean over kept) 卡在 {hist_skip_kept[-1]['acc']:.3f} -> 归一化陷阱放大步长, 加速熵塌缩;")
print(f"无 entropy bonus 卡在 {hist_noent[-1]['acc']:.3f} -> '自信地错'后退化组占满, 梯度枯竭。")

## format reward：先学格式，再学正确

真实 RLVR 中 accuracy reward 的前提是**答案能被提取**：没写 `\boxed{}` 的正确答案，verifier 只能判 0。我们给策略加一个 2 维 **format 头**（是否带规定前缀输出），奖励设计完全照搬 R1 的两件套：

$$r = \underbrace{\mathbb{1}[\text{答案对}] \cdot \mathbb{1}[\text{格式对}]}_{\text{accuracy（格式错则提取失败，判 0）}} \;+\; 0.2\cdot\underbrace{\mathbb{1}[\text{格式对}]}_{\text{format reward}}$$

预期现象（讲解 §4）：format 是单个二值决策、每步都有 0.2 的稠密信号 → **几十步内饱和**；accuracy 信号稀疏 → 慢得多。曲线呈清晰的**两阶段**：先把"答案放在哪"学会，accuracy 的奖励通道才干净起来。

In [ ]:
def train_format(steps=4000, B=128, G=8, beta=0.005, lr=3e-3, ent_coef=0.10, seed=0, every=25):
    torch.manual_seed(seed)
    policy = PolicyNet(n_out=N_ANS + 2)            # 前 199 维: 答案头; 后 2 维: format 头
    ref = copy.deepcopy(policy)
    for p in ref.parameters(): p.requires_grad_(False)
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    hist = []
    for step in range(1, steps + 1):
        a, b = make_questions(B)
        x = features(a, b)
        out = policy(x)
        ans_lp = F.log_softmax(out[:, :N_ANS], -1)
        fmt_lp = F.log_softmax(out[:, N_ANS:], -1)
        with torch.no_grad():
            ans = torch.distributions.Categorical(logits=out[:, :N_ANS]).sample((G,)).T
            fmt = torch.distributions.Categorical(logits=out[:, N_ANS:]).sample((G,)).T
        acc_r = verify_batch(a, b, ans)
        fmt_r = (fmt == 1).float()                 # 1 = 按规定前缀输出
        r = acc_r * fmt_r + 0.2 * fmt_r            # 格式错 -> 提取失败 -> accuracy 记 0
        adv = group_advantage(r)
        logp = ans_lp.gather(1, ans) + fmt_lp.gather(1, fmt)   # 联合 log prob
        with torch.no_grad():
            ro = ref(x)
            logp_ref = (F.log_softmax(ro[:, :N_ANS], -1).gather(1, ans)
                        + F.log_softmax(ro[:, N_ANS:], -1).gather(1, fmt))
        entropy = -(ans_lp.exp() * ans_lp).sum(-1).mean()      # entropy bonus 只加在答案头
        loss = grpo_loss(logp, adv, logp_ref, beta) - ent_coef * entropy
        opt.zero_grad(); loss.backward(); opt.step()
        if step % every == 0:
            hist.append(dict(step=step, acc=acc_r.mean().item(), fmt=fmt_r.mean().item()))
    return hist

hist_fmt = train_format()
plt.figure(figsize=(6.5, 3.2))
plt.plot([h['step'] for h in hist_fmt], [h['fmt'] for h in hist_fmt], label='format compliance')
plt.plot([h['step'] for h in hist_fmt], [h['acc'] for h in hist_fmt], label='answer accuracy (sampled)')
plt.xlabel('step'); plt.ylabel('rate'); plt.legend(); plt.title('two-stage: format first, accuracy later')
plt.tight_layout(); plt.show()
fmt90 = next((h['step'] for h in hist_fmt if h['fmt'] > 0.9), None)
acc50 = next((h['step'] for h in hist_fmt if h['acc'] > 0.5), None)
print(f"format > 90% 在 step {fmt90}; accuracy > 50% 在 step {acc50} —— 两阶段清晰可见")
print("注: 复合奖励下两个头共享同一个组 advantage, format 饱和前 accuracy 的信用分配是被污染的,")
print("    这正是『format reward 权重不能太大』的玩具版证据。")

## 对照基线：拒绝采样微调（RFT / BoN + SFT）

讲解 §7 的"永远先跑的简单基线"：每轮对一批题采 $G$ 个答案，verifier **只留对的**，把 (题目, 正确答案) 对加入缓冲区，然后**从初始模型重训** SFT（STaR 式重训可避免自我强化的模式坍缩——我们实测过：若在当前模型上持续 SFT，策略会坍缩到把同一个答案塞给所有题）。

与 GRPO 的本质区别：**RFT 丢掉全部负样本**（错误答案不产生压低梯度），且离线、无 KL/entropy 正则。下面在**同一任务、按累计采样数对齐**的设定下比较两条曲线。

In [ ]:
def train_rft(rounds=8, n_q=32000, G=8, lr=3e-3, tau=2.0, epochs=15, bs=128, seed=0):
    torch.manual_seed(seed)
    init = PolicyNet()
    policy = copy.deepcopy(init)
    seen, buf_a, buf_b, buf_y = set(), [], [], []
    hist = []
    for rd in range(1, rounds + 1):
        a, b = make_questions(n_q)
        with torch.no_grad():                       # 温度 tau 采样维持多样性
            samp = torch.distributions.Categorical(logits=policy(features(a, b)) / tau).sample((G,)).T
        r = verify_batch(a, b, samp)
        iq, ig = torch.nonzero(r, as_tuple=True)    # 只留 verifier 判对的样本
        for q, g_ in zip(iq.tolist(), ig.tolist()):
            key = (a[q].item(), b[q].item())
            if key not in seen:                     # 每道题只入库一次
                seen.add(key)
                buf_a.append(a[q]); buf_b.append(b[q]); buf_y.append(samp[q, g_])
        if buf_y:
            X = features(torch.stack(buf_a), torch.stack(buf_b))
            Y = torch.stack(buf_y)
            policy = copy.deepcopy(init)            # STaR 式: 每轮从初始模型重训
            opt = torch.optim.Adam(policy.parameters(), lr=lr)
            for _ in range(max(1, epochs * len(Y) // bs)):
                j = torch.randint(0, len(Y), (min(bs, len(Y)),))
                loss = F.cross_entropy(policy(X[j]), Y[j])
                opt.zero_grad(); loss.backward(); opt.step()
        hist.append(dict(samples=rd * n_q * G, acc=greedy_accuracy(policy), buffer=len(buf_y)))
        print(f"RFT round {rd}: 累计采样 {rd*n_q*G:>8d}  buffer {len(buf_y):>5d}  acc {hist[-1]['acc']:.3f}")
    return hist

hist_rft = train_rft()
plt.figure(figsize=(6.5, 3.2))
plt.plot([h['samples'] for h in hist_grpo], [h['acc'] for h in hist_grpo], label='GRPO (B=128, G=8)')
plt.plot([h['samples'] for h in hist_rft], [h['acc'] for h in hist_rft], marker='o', label='RFT (BoN+SFT)')
plt.xlabel('total sampled answers'); plt.ylabel('greedy accuracy')
plt.legend(); plt.title('sample efficiency: GRPO vs rejection sampling FT')
plt.tight_layout(); plt.show()
print(f"\n注意 buffer 已覆盖 {hist_rft[-1]['buffer']} / 8100 道可能的题 ——")
print("RFT 在这个玩具任务上甚至更快, 很大程度因为题空间小到能被 BoN『穷举』并全题库记忆;")
print("真实数学/代码任务题空间不可穷举、SFT 重训成本高, 且 RFT 永远学不到负样本信号。")
print("评测视角: 玩具/小 benchmark 上的方法排序可能与大规模设定相反, 下结论前先检查这类规模混淆。")

## overthinking 玩具：长度成本如何唤回简洁

讲解 §6.1：RLVR 只奖励对错、不计成本，"多想一步"的边际收益恒为正 → 输出长度单调膨胀，简单题也长链。我们用一个最小模型复现：策略只学一个**思考长度** $L\in\{1,\dots,8\}$ 的分布，答对概率随长度饱和递增（模拟"多想有用但边际递减"）：

$$p_{\text{correct}}(L) = 0.30 + 0.65\,(1 - e^{-(L-1)/1.5}), \qquad r = \mathbb{1}[\text{对}] - \lambda L$$

- $\lambda=0$：只要 $p_{\text{correct}}$ 还在涨哪怕一点点，最优策略就是 $L=8$ —— **overthinking**；
- $\lambda>0$：策略缩短到"边际正确率收益 = 边际长度成本"的均衡点。

这正是评测推理模型必须报告 token 预算的玩具版理由：改变 $\lambda$（= 改变对算力的计价），"最优行为"随之改变，**分数只有在固定预算口径下才可比**。

In [ ]:
def p_correct(L):
    return 0.30 + 0.65 * (1 - torch.exp(-(L - 1) / 1.5))

def train_length(lam, steps=600, G=64, lr=0.05, seed=0):
    torch.manual_seed(seed)
    logits = torch.zeros(8, requires_grad=True)        # L in 1..8 的分布
    opt = torch.optim.Adam([logits], lr=lr)
    hist = []
    for step in range(steps):
        dist = torch.distributions.Categorical(logits=logits)
        Ls = dist.sample((G,))
        L = (Ls + 1).float()
        correct = torch.bernoulli(p_correct(L))        # 模拟: 想得越久越可能对
        r = correct - lam * L                          # 长度成本
        adv = (r - r.mean()) / (r.std(unbiased=False) + 1e-4)
        loss = -(adv * dist.log_prob(Ls)).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 25 == 0 or step == steps - 1:
            with torch.no_grad():
                mean_L = (torch.softmax(logits, -1) * torch.arange(1, 9).float()).sum().item()
            hist.append(dict(step=step, mean_len=mean_L))
    return hist

h0 = train_length(lam=0.0)
h1 = train_length(lam=0.05)
plt.figure(figsize=(6.5, 3.2))
plt.plot([h['step'] for h in h0], [h['mean_len'] for h in h0], label='no length cost (lam=0)')
plt.plot([h['step'] for h in h1], [h['mean_len'] for h in h1], label='length cost lam=0.05')
plt.xlabel('step'); plt.ylabel('mean thinking length'); plt.legend(); plt.title('overthinking & length penalty')
plt.tight_layout(); plt.show()
gain = [round((p_correct(torch.tensor(float(l + 1))) - p_correct(torch.tensor(float(l)))).item(), 3) for l in range(1, 8)]
print("每多想一步的边际正确率收益 L->L+1:", gain)
print(f"lam=0    收敛长度 ~ {h0[-1]['mean_len']:.2f} (边际收益>0 就继续想 -> overthinking)")
print(f"lam=0.05 收敛长度 ~ {h1[-1]['mean_len']:.2f} (在边际收益 < 0.05 处停下)")

## ✏️ 练习 1：实现 `group_advantage(rewards)`

实现 GRPO 的组内标准化 advantage（前面用过它，现在**不回看实现**自己写一遍）。

**要求**：输入 `rewards` 形状 `(B, G)`，返回同形状的 advantage：
1. 组内减均值、除以**总体标准差**（`unbiased=False`）；
2. 用 `eps=1e-4` 平滑分母，保证全对/全错的组返回**全 0**（而不是 NaN）。

提示：3 行以内可完成；注意 `keepdim=True`。

In [ ]:
def group_advantage(rewards, eps=1e-4):
    # rewards: (B, G) 的奖励张量
    # TODO: 计算组内均值 mean 与总体标准差 std (dim=-1, keepdim=True, unbiased=False)
    # TODO: 返回 (rewards - mean) / (std + eps)
    raise NotImplementedError

In [ ]:
# ---- 练习 1 自测 ----
adv = group_advantage(torch.tensor([[1., 0., 0., 0.]]))
assert adv.shape == (1, 4), "形状必须保持 (B, G)"
assert abs(adv[0, 0].item() - 3 ** 0.5) < 1e-3, "1 correct of 4: 正样本 advantage 应为 sqrt(3)"
assert abs(adv[0, 1].item() + 1 / 3 ** 0.5) < 1e-3, "负样本 advantage 应为 -1/sqrt(3)"
assert abs(adv.mean().item()) < 1e-6, "组内 advantage 均值应为 0"
deg = group_advantage(torch.tensor([[1., 1., 1., 1.]]))
assert not torch.isnan(deg).any(), "全对组不能出 NaN"
assert torch.allclose(deg, torch.zeros(1, 4)), "全对组 advantage 应为全 0"
assert torch.allclose(group_advantage(torch.tensor([[0., 0., 0., 0.]])), torch.zeros(1, 4)), "全错组同理"
batch = group_advantage(torch.tensor([[1., 0., 1., 0.], [1., 1., 1., 1.]]))
assert abs(batch[0, 0].item() - 1.0) < 1e-3 and torch.allclose(batch[1], torch.zeros(4)), "批内逐组独立计算"
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现容错的 `verifier(question, answer)`

训练里我们用的是张量化精确匹配；真实 RLVR 的 verifier 要先从**自由文本**里提取答案。实现：

```
verifier(question, answer_text) -> 1.0 / 0.0
```

**要求**：`question` 是 `(a, b)` 元组；从 `answer_text` 中提取**最后一个**整数（惯例：最终答案在末尾，对应真实评测中取 `\boxed{}` 或 "the answer is" 之后的内容），与 `a+b` 比对。要能容错处理 `"59"`、`"答案是 59"`、`"  59  "`、`"答案=59"`、`"42+17=59"` 等格式；提取不到任何数字返回 `0.0`。

提示：`re.findall(r"\d+", text)`；为什么取**最后一个**数而不是第一个？想想 `"42+17=59"` 这个用例——这本身就是一个 verifier 设计决策。

In [ ]:
def verifier(question, answer_text):
    # question: (a, b); answer_text: 模型输出字符串
    # TODO: 用 re.findall 提取所有整数; 为空返回 0.0
    # TODO: 取最后一个整数与 a + b 比对, 相等返回 1.0, 否则 0.0
    raise NotImplementedError

In [ ]:
# ---- 练习 2 自测 ----
q = (42, 17)   # 和 = 59
assert verifier(q, "59") == 1.0
assert verifier(q, "答案是 59") == 1.0
assert verifier(q, "  59  ") == 1.0
assert verifier(q, "答案=59") == 1.0
assert verifier(q, "42+17=59") == 1.0, "应取最后一个数字 59, 而不是 42"
assert verifier(q, "59。") == 1.0
assert verifier(q, "58") == 0.0, "答案错误应判 0"
assert verifier(q, "我不知道") == 0.0, "无数字应判 0"
assert verifier(q, "") == 0.0
assert verifier((10, 10), "先算个位, 答案是 20") == 1.0
assert isinstance(verifier(q, "59"), float), "返回 float 奖励"
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 `grpo_loss(logp, advantages, logp_ref, beta)`

实现 GRPO 损失（同样不回看前面的实现）：

$$\mathcal{L} = -\mathrm{mean}(\hat{A} \cdot \log\pi_\theta) + \beta\cdot\mathrm{mean}\Big(\tfrac{\pi_{\text{ref}}}{\pi_\theta} - \log\tfrac{\pi_{\text{ref}}}{\pi_\theta} - 1\Big)$$

**要求**：
1. policy gradient 项：`-(advantages * logp).mean()`；
2. KL 项用 $k_3$ 估计器：令 `log_ratio = logp_ref - logp`，则 `kl = (log_ratio.exp() - log_ratio - 1).mean()`；
3. 返回 `pg + beta * kl`。

自测会检查**梯度方向**（正 advantage 必须推高 logp）和 KL 项的贡献（恒非负、两策略相同时为 0）。

In [ ]:
def grpo_loss(logp, advantages, logp_ref, beta):
    # logp / logp_ref / advantages: 同形状张量
    # TODO: pg = -(advantages * logp).mean()
    # TODO: k3 KL: log_ratio = logp_ref - logp; kl = (log_ratio.exp() - log_ratio - 1).mean()
    # TODO: 返回 pg + beta * kl
    raise NotImplementedError

In [ ]:
# ---- 练习 3 自测 ----
# (1) 数值: adv=1, logp=-1, ref 相同, beta=0 -> loss = 1.0
lp = torch.tensor([-1.0])
assert abs(grpo_loss(lp, torch.tensor([1.0]), torch.tensor([-1.0]), beta=0.0).item() - 1.0) < 1e-6
# (2) 梯度方向: 正 advantage -> d loss / d logp < 0 (梯度下降会推高 logp)
lp_pos = torch.tensor([-1.0], requires_grad=True)
grpo_loss(lp_pos, torch.tensor([2.0]), torch.tensor([-1.0]), beta=0.0).backward()
assert lp_pos.grad.item() < 0, "正 advantage 必须推高被采样答案的 logp"
lp_neg = torch.tensor([-1.0], requires_grad=True)
grpo_loss(lp_neg, torch.tensor([-2.0]), torch.tensor([-1.0]), beta=0.0).backward()
assert lp_neg.grad.item() > 0, "负 advantage 必须压低被采样答案的 logp"
# (3) KL 项: 两策略相同时为 0; 不同时恒为正贡献
same = grpo_loss(lp, torch.tensor([1.0]), torch.tensor([-1.0]), beta=1.0)
assert torch.allclose(same, grpo_loss(lp, torch.tensor([1.0]), torch.tensor([-1.0]), beta=0.0)), "pi=pi_ref 时 KL=0"
diff_on = grpo_loss(lp, torch.tensor([1.0]), torch.tensor([-2.0]), beta=1.0)
diff_off = grpo_loss(lp, torch.tensor([1.0]), torch.tensor([-2.0]), beta=0.0)
assert diff_on > diff_off, "k3 估计的 KL 恒 >= 0, beta>0 时应增大 loss"
print("✅ 练习 3 通过")

## 📖 参考答案

先自己做，再对照。每题一个 cell。

In [ ]:
# ---- 练习 1 参考答案（先自己做，再对照）----
def group_advantage(rewards, eps=1e-4):
    mean = rewards.mean(dim=-1, keepdim=True)
    std = rewards.std(dim=-1, unbiased=False, keepdim=True)
    return (rewards - mean) / (std + eps)
# 要点: 分子 r - mean 在退化组恒为 0, 所以 eps 平滑后整组 advantage = 0, 自动实现"零梯度跳过"。

In [ ]:
# ---- 练习 2 参考答案（先自己做，再对照）----
def verifier(question, answer_text):
    a, b = question
    nums = re.findall(r"\d+", answer_text)
    if not nums:
        return 0.0
    return 1.0 if int(nums[-1]) == a + b else 0.0
# 要点: 取最后一个数对应"最终答案在末尾"的惯例; 若取第一个, "42+17=59" 会被误判错。
# 这类提取规则本身就是 hack 面: 宽松匹配会让模型学会罗列候选答案(讲解 §1/§4)。

In [ ]:
# ---- 练习 3 参考答案（先自己做，再对照）----
def grpo_loss(logp, advantages, logp_ref, beta):
    pg = -(advantages * logp).mean()
    log_ratio = logp_ref - logp
    kl = (log_ratio.exp() - log_ratio - 1).mean()
    return pg + beta * kl
# 要点: k3 = exp(x) - x - 1 >= 0 (x = log_ratio), 是 KL(pi_theta || pi_ref) 的无偏估计;
# 相比直接用 -x (k1, 无偏但方差大、可正可负), k3 在每个样本上都非负, 训练更稳。

## 小结

| 实验 | 你应该带走的结论 |
|---|---|
| GRPO 主循环 | 无 critic 也能训：组均值就是 baseline；准确率 0.5% → >90%，全程只有 policy + 冻结 ref |
| std=0 退化 | naive 除 std 会 NaN 毒化 batch；ε 与 skip（正确归一化下）等价；skip 后对保留样本取 mean 是会放大有效步长的实现陷阱；真正的解药是难度课程与维持探索 |
| entropy collapse | 没有探索来源时，策略"自信地错"后全组同答案 → 零梯度 → 永久卡死；真实 RLVR 靠预训练先验 + 熵监控 |
| format reward | 稠密的格式信号先饱和、稀疏的正确性信号后起飞——两阶段曲线；格式是 accuracy 奖励通道的"信噪比开关" |
| RFT 对比 | 简单基线未必输；但本玩具中 RFT 的强势来自"题空间可穷举"——把规模混淆当成方法优势是评测大忌 |
| overthinking | 不给长度计价，"多想"的边际收益恒为正 → 长度膨胀；改变计价就改变最优行为 → **评测推理模型必须固定并报告 token 预算** |

**下一章** → [06 · 对齐的评测](../06_alignment_evals/06_讲解.html)：跳出"怎么训"，系统回答"怎么知道对齐有没有成功"——LLM-as-a-judge、长度偏置、sycophancy 与 alignment tax 的测量。

---
## 🎯 真实数据胶囊题：真实 GSM8K 上的可验证奖励与 GRPO 优势

RLVR 用**可验证**奖励（答案对不对）。从真实 GSM8K 解析金标答案（#### 后的数），实现精确匹配奖励 + GRPO 的组内归一化优势 `(r-mean)/std`。

> 本模块新增的**真实数据**练习：自包含、用真实公开数据把本章技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.post_training_data"); os.makedirs(CACHE,exist_ok=True)
def _fetch(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=200):
    p=_fetch("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    rows=[json.loads(l) for l in open(p).read().splitlines()[:n]]
    return rows
def winequality():
    import pandas as pd
    p=_fetch("https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv","winequality-red.csv")
    return pd.read_csv(p, sep=";")

rows = gsm8k(100)
def gold(ans):
    return ans.split("####")[-1].strip().replace(",","")
print("真实金标答案示例:", [gold(r["answer"]) for r in rows[:3]])

**练习**：实现 `extract_answer(text)`（取 `####` 后的数字串，没有则取最后一个数字）和 `grpo_advantage(rewards)`（组内 `(r-mean)/(std+1e-8)`）。

In [ ]:
def extract_answer(text):
    # TODO: 优先 '####' 后内容；否则正则找最后一个数字
    raise NotImplementedError
def grpo_advantage(rewards):
    # TODO: (r - mean)/(std+1e-8)
    raise NotImplementedError


In [ ]:
# 自测
assert extract_answer("blah #### 42")=="42"
assert extract_answer("the answer is 17")=="17"
# 真实 GSM8K：解析金标与 gold() 一致
assert extract_answer(rows[0]["answer"])==gold(rows[0]["answer"])
# GRPO 优势：一组奖励里全对得 0 优势(无信号)，有对有错才有信号
adv = grpo_advantage(np.array([1.,0.,1.,0.]))
assert abs(adv.mean())<1e-6 and adv[0]>0 and adv[1]<0
assert np.allclose(grpo_advantage(np.array([1.,1.,1.])), 0), "全对时组内无优势信号"
print("RLVR 可验证奖励 + GRPO 优势 ✓")


### 📖 参考答案

In [ ]:
def extract_answer(text):
    if "####" in text: return text.split("####")[-1].strip().replace(",","")
    nums=re.findall(r"-?\d+\.?\d*", text); return nums[-1] if nums else None
def grpo_advantage(rewards):
    r=np.asarray(rewards,float); return (r-r.mean())/(r.std()+1e-8)
print("✓ 可验证奖励 + 组内归一化 = GRPO/R1 的核心，无需 reward model")